# ECG Project — Phase 1: MIT-BIH Signal Baseline

This notebook validates the uploaded datasets, visualizes heartbeat samples, trains a five-class MIT-BIH baseline model, evaluates it, and saves the model as a pickle-compatible Joblib file.

> Important: this model accepts **187 ECG waveform values**, not values extracted from a printed ECG PDF. The PDF-parameter pipeline will be built separately.

In [ ]:
!pip install -q pandas numpy matplotlib scikit-learn joblib


## 1. Upload and extract the four ZIP files

Upload:
- `mitbih_train.zip`
- `mitbih_test.csv.zip`
- `ptbdb_normal.csv.zip`
- `ptbdb_abnormal.csv.zip`

In [ ]:
from google.colab import files
import zipfile
import os

uploaded = files.upload()
os.makedirs('/content/ecg_data', exist_ok=True)

for filename in uploaded:
    if filename.lower().endswith('.zip'):
        with zipfile.ZipFile(filename, 'r') as zf:
            zf.extractall('/content/ecg_data')

print('Extracted files:')
for root, _, names in os.walk('/content/ecg_data'):
    for name in names:
        print(os.path.join(root, name))


## 2. Locate and load the CSV files

In [ ]:
from pathlib import Path
import pandas as pd

data_dir = Path('/content/ecg_data')

def find_csv(keyword):
    matches = [p for p in data_dir.rglob('*.csv') if keyword.lower() in p.name.lower()]
    if not matches:
        raise FileNotFoundError(f'Could not find a CSV containing: {keyword}')
    return matches[0]

paths = {
    'mit_train': find_csv('mitbih_train'),
    'mit_test': find_csv('mitbih_test'),
    'ptb_normal': find_csv('ptbdb_normal'),
    'ptb_abnormal': find_csv('ptbdb_abnormal'),
}

datasets = {name: pd.read_csv(path, header=None) for name, path in paths.items()}

for name, df in datasets.items():
    print(f'{name:14s} shape={df.shape}, missing={int(df.isna().sum().sum())}')
    print('labels:', df.iloc[:, -1].value_counts().sort_index().to_dict())


## 3. Validate the expected format

Every row should contain 187 waveform samples and one label.

In [ ]:
for name, df in datasets.items():
    assert df.shape[1] == 188, f'{name} has {df.shape[1]} columns instead of 188.'
    assert not df.isna().any().any(), f'{name} contains missing values.'

assert set(datasets['mit_train'].iloc[:, -1].astype(int).unique()) == {0, 1, 2, 3, 4}
assert set(datasets['mit_test'].iloc[:, -1].astype(int).unique()) == {0, 1, 2, 3, 4}
assert set(datasets['ptb_normal'].iloc[:, -1].astype(int).unique()) == {0}
assert set(datasets['ptb_abnormal'].iloc[:, -1].astype(int).unique()) == {1}

print('Dataset validation passed.')


## 4. Visualize one MIT-BIH heartbeat from each class

In [ ]:
import matplotlib.pyplot as plt

mit_train = datasets['mit_train']
label_names = {
    0: 'Normal',
    1: 'Supraventricular ectopic',
    2: 'Ventricular ectopic',
    3: 'Fusion',
    4: 'Unknown / paced / unclassifiable',
}

for label, name in label_names.items():
    row = mit_train[mit_train.iloc[:, -1].astype(int) == label].iloc[0, :-1]
    plt.figure(figsize=(10, 3))
    plt.plot(row.values)
    plt.title(f'Class {label}: {name}')
    plt.xlabel('Sample index')
    plt.ylabel('Normalized amplitude')
    plt.grid(True)
    plt.show()


## 5. Prepare training and test arrays

In [ ]:
import numpy as np

X_train = datasets['mit_train'].iloc[:, :-1].to_numpy(dtype=np.float32)
y_train = datasets['mit_train'].iloc[:, -1].to_numpy(dtype=np.int64)
X_test = datasets['mit_test'].iloc[:, :-1].to_numpy(dtype=np.float32)
y_test = datasets['mit_test'].iloc[:, -1].to_numpy(dtype=np.int64)

print('X_train:', X_train.shape)
print('X_test :', X_test.shape)


## 6. Train an Extra Trees baseline

`class_weight='balanced'` helps reduce the effect of the large class imbalance.

In [ ]:
from sklearn.ensemble import ExtraTreesClassifier

model = ExtraTreesClassifier(
    n_estimators=250,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
    max_features='sqrt'
)

model.fit(X_train, y_train)
print('Training completed.')


## 7. Evaluate the model

Do not judge the model only by overall accuracy. Check recall and F1-score for every class.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, balanced_accuracy_score
from sklearn.metrics import ConfusionMatrixDisplay

pred = model.predict(X_test)

print('Balanced accuracy:', balanced_accuracy_score(y_test, pred))
print('\nClassification report:\n')
print(classification_report(
    y_test,
    pred,
    labels=[0, 1, 2, 3, 4],
    target_names=[label_names[i] for i in range(5)],
    digits=4,
    zero_division=0
))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    pred,
    labels=[0, 1, 2, 3, 4],
    display_labels=['N', 'S', 'V', 'F', 'Q'],
    normalize='true',
    xticks_rotation=45
)
plt.title('Normalized confusion matrix')
plt.show()


## 8. Save the model and metadata

In [ ]:
import joblib

artifact = {
    'model': model,
    'label_names': label_names,
    'input_features': 187,
    'input_type': 'normalized_heartbeat_waveform',
    'dataset': 'MIT-BIH preprocessed CSV',
}

joblib.dump(artifact, '/content/mitbih_signal_baseline.pkl', compress=3)
print('Saved: /content/mitbih_signal_baseline.pkl')


## 9. Reload and test the saved pickle file

In [ ]:
loaded = joblib.load('/content/mitbih_signal_baseline.pkl')
sample = X_test[[0]]
sample_prediction = int(loaded['model'].predict(sample)[0])
sample_probability = float(loaded['model'].predict_proba(sample).max())

print('Predicted:', loaded['label_names'][sample_prediction])
print('Confidence:', round(sample_probability, 4))
print('Actual:', loaded['label_names'][int(y_test[0])])


## 10. Download the model

In [ ]:
from google.colab import files
files.download('/content/mitbih_signal_baseline.pkl')


## Next phase

The final application needs a separate dataset whose columns match printed ECG report measurements such as heart rate, PR interval, QRS duration, QT, QTc and axes. These values cannot be reliably reconstructed from isolated 187-point heartbeat rows. We will therefore create the PDF extraction schema first and then choose or build a medically compatible parameter-labelled dataset.